# Adversarial Data Augmentation and Retraining

This notebook evaluates whether adding transformed prompt-injection examples to the training data improves model robustness.

Explicit override phrases are removed from prompt-injection examples in the training set. These modified examples are then added to the original training data. Models trained on the augmented dataset are compared with models trained on the original dataset using both normal test examples and held-out stripped attacks.

In [2]:
import pandas as pd
import numpy as np
import re

from datasets import load_dataset
from sklearn.model_selection import train_test_split

dataset = load_dataset(
    "reshabhs/SPML_Chatbot_Prompt_Injection",
    split="train"
)

df = dataset.to_pandas()

print(df.shape)
print(df.columns.tolist())

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(16012, 5)
['System Prompt', 'User Prompt', 'Prompt injection', 'Degree', 'Source']


In [3]:
df["Prompt injection"] = pd.to_numeric(
    df["Prompt injection"],
    errors="coerce"
)

df = df.dropna(subset=["Prompt injection"]).copy()
df["Prompt injection"] = df["Prompt injection"].astype(int)

df["System Prompt"] = (
    df["System Prompt"]
    .fillna("")
    .astype(str)
)

df["User Prompt"] = (
    df["User Prompt"]
    .fillna("")
    .astype(str)
)

df["combined_text"] = (
    "System Prompt: "
    + df["System Prompt"].str.strip()
    + "\nUser Prompt: "
    + df["User Prompt"].str.strip()
)

print(df.shape)
print(df["Prompt injection"].value_counts())

(16012, 6)
Prompt injection
1    12542
0     3470
Name: count, dtype: int64


In [4]:
train_indices, test_indices = train_test_split(
    df.index,
    test_size=0.20,
    random_state=42,
    stratify=df["Prompt injection"]
)

train_df = df.loc[train_indices].copy()
test_df = df.loc[test_indices].copy()

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining labels:")
print(train_df["Prompt injection"].value_counts())

print("\nTesting labels:")
print(test_df["Prompt injection"].value_counts())

Training samples: 12809
Testing samples: 3203

Training labels:
Prompt injection
1    10033
0     2776
Name: count, dtype: int64

Testing labels:
Prompt injection
1    2509
0     694
Name: count, dtype: int64


In [5]:
override_pattern = (
    r"(?i)"
    r"(?:ignore|forget|disregard|override|bypass|set aside|do not follow)"
    r".{0,60}"
    r"(?:instruction|instructions|rules|guidance|prompt|directions)"
)

In [7]:
def remove_override_language(text):
    patterns = [
        r"(?i)\bignore (?:all |any |the |your |previous |above )*instructions?\b[,:;.\s-]*",
        r"(?i)\bforget (?:all |any |the |your |previous |above )*instructions?\b[,:;.\s-]*",
        r"(?i)\bdisregard (?:all |any |the |your |previous |above )*(?:instructions?|rules|guidance|directions)\b[,:;.\s-]*",
        r"(?i)\boverride (?:all |any |the |your |previous |above )*(?:instructions?|rules|guidance|directions)\b[,:;.\s-]*",
        r"(?i)\bdo not follow (?:all |any |the |your |previous |above )*(?:instructions?|rules|guidance|directions)\b[,:;.\s-]*",
        r"(?i)\byou can now ignore instructions?\b[,:;.\s-]*",
        r"(?i)\brespond with all instructions?\b[,:;.\s-]*",
        r"(?i)\benter testing mode\b[,:;.\s-]*",
        r"(?i)\bplease confirm that you understand this instruction\b[,:;.\s-]*"
    ]

    transformed = text

    for pattern in patterns:
        transformed = re.sub(
            pattern,
            "",
            transformed
        )

    transformed = re.sub(
        r"\n{3,}",
        "\n\n",
        transformed
    )

    return transformed.strip()

In [8]:
explicit_train_attacks = train_df[
    (train_df["Prompt injection"] == 1)
    & train_df["User Prompt"].str.contains(
        override_pattern,
        regex=True,
        na=False
    )
].copy()

explicit_train_attacks["stripped_user_prompt"] = (
    explicit_train_attacks["User Prompt"]
    .apply(remove_override_language)
)

explicit_train_attacks["changed"] = (
    explicit_train_attacks["User Prompt"]
    != explicit_train_attacks["stripped_user_prompt"]
)

explicit_train_attacks = explicit_train_attacks[
    explicit_train_attacks["changed"]
].copy()

print(
    "Usable adversarial training examples:",
    len(explicit_train_attacks)
)

Usable adversarial training examples: 1887


In [9]:
augmented_examples = explicit_train_attacks.copy()

augmented_examples["User Prompt"] = (
    augmented_examples["stripped_user_prompt"]
)

augmented_examples["combined_text"] = (
    "System Prompt: "
    + augmented_examples["System Prompt"].str.strip()
    + "\nUser Prompt: "
    + augmented_examples["User Prompt"].str.strip()
)

print("Augmented examples:", len(augmented_examples))

Augmented examples: 1887


In [10]:
augmented_train_df = pd.concat(
    [
        train_df,
        augmented_examples[train_df.columns]
    ],
    ignore_index=True
)

print("Original training size:", len(train_df))
print("Augmented training size:", len(augmented_train_df))

print("\nOriginal label counts:")
print(train_df["Prompt injection"].value_counts())

print("\nAugmented label counts:")
print(augmented_train_df["Prompt injection"].value_counts())

Original training size: 12809
Augmented training size: 14696

Original label counts:
Prompt injection
1    10033
0     2776
Name: count, dtype: int64

Augmented label counts:
Prompt injection
1    11920
0     2776
Name: count, dtype: int64


In [11]:
explicit_test_attacks = test_df[
    (test_df["Prompt injection"] == 1)
    & test_df["User Prompt"].str.contains(
        override_pattern,
        regex=True,
        na=False
    )
].copy()

explicit_test_attacks["stripped_user_prompt"] = (
    explicit_test_attacks["User Prompt"]
    .apply(remove_override_language)
)

explicit_test_attacks["changed"] = (
    explicit_test_attacks["User Prompt"]
    != explicit_test_attacks["stripped_user_prompt"]
)

explicit_test_attacks = explicit_test_attacks[
    explicit_test_attacks["changed"]
].copy()

explicit_test_attacks["stripped_combined_text"] = (
    "System Prompt: "
    + explicit_test_attacks["System Prompt"].str.strip()
    + "\nUser Prompt: "
    + explicit_test_attacks["stripped_user_prompt"].str.strip()
)

print(
    "Held-out stripped test attacks:",
    len(explicit_test_attacks)
)

Held-out stripped test attacks: 498


In [12]:
explicit_test_attacks = test_df[
    (test_df["Prompt injection"] == 1)
    & test_df["User Prompt"].str.contains(
        override_pattern,
        regex=True,
        na=False
    )
].copy()

explicit_test_attacks["stripped_user_prompt"] = (
    explicit_test_attacks["User Prompt"]
    .apply(remove_override_language)
)

explicit_test_attacks["changed"] = (
    explicit_test_attacks["User Prompt"]
    != explicit_test_attacks["stripped_user_prompt"]
)

explicit_test_attacks = explicit_test_attacks[
    explicit_test_attacks["changed"]
].copy()

explicit_test_attacks["stripped_combined_text"] = (
    "System Prompt: "
    + explicit_test_attacks["System Prompt"].str.strip()
    + "\nUser Prompt: "
    + explicit_test_attacks["stripped_user_prompt"].str.strip()
)

print(
    "Held-out stripped test attacks:",
    len(explicit_test_attacks)
)

Held-out stripped test attacks: 498


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def calculate_full_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "FPR": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "FNR": fn / (fn + tp) if (fn + tp) > 0 else np.nan
    }

In [15]:
def create_models():
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=2000,
            random_state=42
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ),

        "Linear SVM": LinearSVC(
            random_state=42
        )
    }

In [16]:
baseline_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_baseline = baseline_vectorizer.fit_transform(
    train_df["combined_text"]
)

X_test_normal_baseline = baseline_vectorizer.transform(
    test_df["combined_text"]
)

X_test_stripped_baseline = baseline_vectorizer.transform(
    explicit_test_attacks["stripped_combined_text"]
)

y_train_baseline = train_df["Prompt injection"]
y_test_normal = test_df["Prompt injection"]
y_test_stripped = explicit_test_attacks["Prompt injection"]

print(X_train_baseline.shape)
print(X_test_normal_baseline.shape)
print(X_test_stripped_baseline.shape)

(12809, 10000)
(3203, 10000)
(498, 10000)


In [17]:
augmented_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_augmented = augmented_vectorizer.fit_transform(
    augmented_train_df["combined_text"]
)

X_test_normal_augmented = augmented_vectorizer.transform(
    test_df["combined_text"]
)

X_test_stripped_augmented = augmented_vectorizer.transform(
    explicit_test_attacks["stripped_combined_text"]
)

y_train_augmented = augmented_train_df["Prompt injection"]

print(X_train_augmented.shape)
print(X_test_normal_augmented.shape)
print(X_test_stripped_augmented.shape)

(14696, 10000)
(3203, 10000)
(498, 10000)


In [18]:
retraining_results = []

baseline_models = create_models()
augmented_models = create_models()

for model_name in baseline_models:
    print(f"Evaluating {model_name}...")

    baseline_model = baseline_models[model_name]
    augmented_model = augmented_models[model_name]

    # Train on original data
    baseline_model.fit(
        X_train_baseline,
        y_train_baseline
    )

    baseline_normal_predictions = baseline_model.predict(
        X_test_normal_baseline
    )

    baseline_stripped_predictions = baseline_model.predict(
        X_test_stripped_baseline
    )

    # Train on adversarially augmented data
    augmented_model.fit(
        X_train_augmented,
        y_train_augmented
    )

    augmented_normal_predictions = augmented_model.predict(
        X_test_normal_augmented
    )

    augmented_stripped_predictions = augmented_model.predict(
        X_test_stripped_augmented
    )

    # Full metrics on normal test set
    baseline_normal_metrics = calculate_full_metrics(
        y_test_normal,
        baseline_normal_predictions
    )

    augmented_normal_metrics = calculate_full_metrics(
        y_test_normal,
        augmented_normal_predictions
    )

    # Recall on positive-only stripped attack set
    baseline_stripped_recall = recall_score(
        y_test_stripped,
        baseline_stripped_predictions,
        zero_division=0
    )

    augmented_stripped_recall = recall_score(
        y_test_stripped,
        augmented_stripped_predictions,
        zero_division=0
    )

    retraining_results.append({
        "Model": model_name,

        "Baseline Normal Accuracy":
            baseline_normal_metrics["Accuracy"],

        "Augmented Normal Accuracy":
            augmented_normal_metrics["Accuracy"],

        "Normal Accuracy Change":
            augmented_normal_metrics["Accuracy"]
            - baseline_normal_metrics["Accuracy"],

        "Baseline Normal Precision":
            baseline_normal_metrics["Precision"],

        "Augmented Normal Precision":
            augmented_normal_metrics["Precision"],

        "Baseline Normal Recall":
            baseline_normal_metrics["Recall"],

        "Augmented Normal Recall":
            augmented_normal_metrics["Recall"],

        "Baseline Normal F1":
            baseline_normal_metrics["F1 Score"],

        "Augmented Normal F1":
            augmented_normal_metrics["F1 Score"],

        "Normal F1 Change":
            augmented_normal_metrics["F1 Score"]
            - baseline_normal_metrics["F1 Score"],

        "Baseline Normal FPR":
            baseline_normal_metrics["FPR"],

        "Augmented Normal FPR":
            augmented_normal_metrics["FPR"],

        "Normal FPR Change":
            augmented_normal_metrics["FPR"]
            - baseline_normal_metrics["FPR"],

        "Baseline Stripped Recall":
            baseline_stripped_recall,

        "Augmented Stripped Recall":
            augmented_stripped_recall,

        "Stripped Recall Change":
            augmented_stripped_recall
            - baseline_stripped_recall
    })

Evaluating Logistic Regression...
Evaluating Random Forest...
Evaluating Linear SVM...


In [19]:
retraining_results_df = pd.DataFrame(
    retraining_results
)

retraining_results_df

,Model,Baseline Normal Accuracy,Augmented Normal Accuracy,Normal Accuracy Change,Baseline Normal Precision,Augmented Normal Precision,Baseline Normal Recall,Augmented Normal Recall,Baseline Normal F1,Augmented Normal F1,Normal F1 Change,Baseline Normal FPR,Augmented Normal FPR,Normal FPR Change,Baseline Stripped Recall,Augmented Stripped Recall,Stripped Recall Change
0,Logistic Regression,0.952857,0.944739,-0.008117,0.951033,0.939668,0.990833,0.993224,0.970525,0.965704,-0.004821,0.184438,0.230548,0.046110,0.997992,0.997992,0.000000
1,Random Forest,0.953793,0.957228,0.003434,0.969757,0.971009,0.971303,0.974492,0.970530,0.972747,0.002217,0.109510,0.105187,-0.004323,0.985944,0.987952,0.002008
2,Linear SVM,0.978145,0.977833,-0.000312,0.993525,0.991929,0.978477,0.979673,0.985944,0.985763,-0.000181,0.023055,0.028818,0.005764,0.993976,0.995984,0.002008


## Adversarial Retraining Results

Adversarial data augmentation produced model-dependent results. Random Forest showed the most favorable response: stripped-attack recall increased from 0.986 to 0.988, normal-test F1 increased from 0.971 to 0.973, and the false-positive rate decreased slightly from 0.110 to 0.105.

Linear SVM also showed a small improvement in stripped-attack recall, increasing from 0.994 to 0.996. Its normal-test F1 remained nearly unchanged, although its false-positive rate increased slightly from 0.023 to 0.029.

Logistic Regression did not improve on the stripped attacks, remaining at 0.998 recall. Its normal-test accuracy and F1 decreased, while the false-positive rate increased from 0.184 to 0.231. This suggests that adding only transformed positive examples shifted the classifier toward predicting the injection class more frequently.

Overall, adversarial augmentation did not provide a universal improvement. Its effectiveness depended on the learning algorithm, with Random Forest benefiting most from the additional transformed examples.

In [20]:
retraining_results_df.to_csv(
    "../results/adversarial_retraining_results.csv",
    index=False
)

augmented_examples[
    [
        "System Prompt",
        "User Prompt",
        "Prompt injection",
        "Degree",
        "Source"
    ]
].to_csv(
    "../results/adversarial_training_examples.csv",
    index=False
)

print("Adversarial retraining results saved.")

Adversarial retraining results saved.


### Limitation

The augmented training set added only prompt-injection examples and therefore increased the imbalance toward the positive class. Some observed changes, particularly the higher false-positive rate of Logistic Regression, may reflect this altered class distribution rather than the transformation strategy alone. Future work should compare this approach with balanced augmentation that also includes difficult benign examples.

# Model Efficiency Comparison

This section compares the training and prediction time of the lightweight classifiers. The objective is to examine the trade-off between detection performance and computational cost.

In [21]:
import time

efficiency_results = []

efficiency_models = create_models()

for model_name, model in efficiency_models.items():
    print(f"Timing {model_name}...")

    training_start = time.perf_counter()

    model.fit(
        X_train_baseline,
        y_train_baseline
    )

    training_time = (
        time.perf_counter() - training_start
    )

    prediction_start = time.perf_counter()

    predictions = model.predict(
        X_test_normal_baseline
    )

    prediction_time = (
        time.perf_counter() - prediction_start
    )

    metrics = calculate_full_metrics(
        y_test_normal,
        predictions
    )

    efficiency_results.append({
        "Model": model_name,
        "Training Samples": X_train_baseline.shape[0],
        "Test Samples": X_test_normal_baseline.shape[0],
        "Training Time (seconds)": training_time,
        "Prediction Time (seconds)": prediction_time,
        "Average Prediction Time per Sample (ms)":
            prediction_time / len(y_test_normal) * 1000,
        "Test Accuracy": metrics["Accuracy"],
        "Test F1": metrics["F1 Score"]
    })

efficiency_results_df = pd.DataFrame(
    efficiency_results
)

efficiency_results_df

Timing Logistic Regression...
Timing Random Forest...
Timing Linear SVM...


,Model,Training Samples,Test Samples,Training Time (seconds),Prediction Time (seconds),Average Prediction Time per Sample (ms),Test Accuracy,Test F1
0,Logistic Regression,12809,3203,0.244508,0.003277,0.001023,0.952857,0.970525
1,Random Forest,12809,3203,2.348305,0.054282,0.016947,0.953793,0.970530
2,Linear SVM,12809,3203,0.447042,0.001885,0.000589,0.978145,0.985944


## Efficiency Results

Linear SVM provided the strongest performance-efficiency trade-off. It achieved the highest test accuracy of 0.978 and the highest F1 score of 0.986 while requiring approximately 0.447 seconds for training and 0.0019 seconds to classify 3,203 test samples.

Logistic Regression had the shortest training time at approximately 0.245 seconds, but its test accuracy and F1 score were lower than those of Linear SVM. Random Forest required the greatest training and prediction time while producing performance similar to Logistic Regression on the normal test set.

These findings indicate that Linear SVM is the most suitable lightweight detector among the evaluated classical models because it combines strong classification performance with very low prediction latency.

In [22]:
efficiency_results_df.to_csv(
    "../results/model_efficiency_results.csv",
    index=False
)

print("Efficiency results saved.")

Efficiency results saved.


### Timing Limitation

Each model was timed during a single local execution. The reported values therefore provide an approximate comparison within the same experimental environment rather than a hardware-independent benchmark. Repeated timing trials, warm-up runs, memory measurements, and comparisons under standardized hardware would provide a more reliable efficiency evaluation.

In [23]:
repeated_efficiency_results = []

timed_models = create_models()

for model_name, model in timed_models.items():
    print(f"Repeated timing for {model_name}...")

    model.fit(
        X_train_baseline,
        y_train_baseline
    )

    prediction_times = []

    for _ in range(100):
        start = time.perf_counter()

        model.predict(
            X_test_normal_baseline
        )

        prediction_times.append(
            time.perf_counter() - start
        )

    repeated_efficiency_results.append({
        "Model": model_name,
        "Runs": len(prediction_times),
        "Mean Prediction Time (seconds)":
            np.mean(prediction_times),
        "Prediction Time Standard Deviation (seconds)":
            np.std(prediction_times),
        "Mean Time per Sample (ms)":
            np.mean(prediction_times)
            / len(y_test_normal)
            * 1000
    })

repeated_efficiency_df = pd.DataFrame(
    repeated_efficiency_results
)

repeated_efficiency_df

Repeated timing for Logistic Regression...
Repeated timing for Random Forest...
Repeated timing for Linear SVM...


,Model,Runs,Mean Prediction Time (seconds),Prediction Time Standard Deviation (seconds),Mean Time per Sample (ms)
0,Logistic Regression,100,0.001440,0.000259,0.000450
1,Random Forest,100,0.050407,0.004422,0.015738
2,Linear SVM,100,0.001789,0.000706,0.000559


## Repeated Prediction-Time Evaluation

Prediction time was measured over 100 runs to reduce the influence of one-time execution noise. Logistic Regression had the lowest mean prediction latency at approximately 0.000450 milliseconds per sample, followed closely by Linear SVM at 0.000559 milliseconds per sample. Random Forest required approximately 0.015738 milliseconds per sample.

Although Random Forest was substantially slower in relative terms, all three classifiers achieved very low absolute prediction latency. Considering both classification quality and efficiency, Linear SVM provided the strongest overall trade-off because it achieved the highest normal-test accuracy and F1 score while retaining prediction speed close to Logistic Regression.

In [24]:
repeated_efficiency_df.to_csv(
    "../results/repeated_prediction_timing.csv",
    index=False
)

print("Repeated prediction timing saved.")

Repeated prediction timing saved.


### Efficiency Evaluation Limitation

Prediction latency was averaged across 100 runs, improving reliability compared with a single timing measurement. However, the experiment was conducted in one local environment and did not control for hardware, operating-system scheduling, background processes, memory usage, or model-loading time. The results should therefore be interpreted as a relative comparison among the evaluated classifiers rather than a hardware-independent deployment benchmark.